# Deep Learning Training — CIC-IDS-2018 (Fixed Pipeline)

**Same methodology as ML Fixed Pipeline notebook.**

| # | Change | Details |
|---|--------|---------|
| 1 | **Corrupt rows removed** | Negative values in time/duration features are physically impossible (CICFlowMeter bug) |
| 2 | **Correct scaling order** | Split train/test FIRST → fit scalers on train only → transform both |
| 3 | **Honest metrics** | Balanced Accuracy + per-class Recall added alongside standard metrics |

> **Note:** Duplicate removal and class balancing are intentionally NOT applied here.
> Each optimization is studied independently as per the research methodology.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import time
import pickle
import warnings
warnings.filterwarnings('ignore')
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    balanced_accuracy_score
)

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import (Dense, Dropout, Conv1D, MaxPooling1D, Flatten,
                          LSTM, BatchNormalization)
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.optimizers import Adam

import matplotlib.pyplot as plt
import seaborn as sns

print('=' * 80)
print('🚀 DEEP LEARNING TRAINING — CIC-IDS-2018 FIXED PIPELINE')
print('=' * 80)

## 2. Configuration

In [ ]:
INPUT_FILE       = 'archive/full_df_binary_labels.csv'
BASE_MODEL_DIR   = 'trained_models/dl_fixed/'
TEST_SIZE        = 0.2
VALIDATION_SPLIT = 0.2
RANDOM_STATE     = 42
SAVE_MODELS      = True

# Training parameters
EPOCHS     = 12
BATCH_SIZE = 256
PATIENCE   = 10

# Per-model dataset size control (same as original)
MODEL_SAMPLE_SIZES = {
    'CNN' : 1000000,
    'LSTM': 200000
}

# Features that are physically impossible to be negative
NON_NEGATIVE_FEATURES = [
    'Flow Duration', 'Flow IAT Mean', 'Flow IAT Std',
    'Flow IAT Max',  'Flow IAT Min',  'Fwd IAT Tot',
    'Fwd IAT Mean',  'Fwd IAT Std',   'Fwd IAT Max',
    'Fwd IAT Min',   'Bwd IAT Tot',   'Bwd IAT Mean',
    'Bwd IAT Std',   'Bwd IAT Max',   'Bwd IAT Min',
    'Idle Mean',     'Idle Max',      'Idle Min',
    'Tot Fwd Pkts',  'Tot Bwd Pkts',  'TotLen Fwd Pkts',
    'TotLen Bwd Pkts', 'Flow Byts/s', 'Flow Pkts/s'
]

# Features to apply PowerTransformer (extreme skew)
POWER_TRANSFORMER_FEATURES = [
    'Dst Port', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts',
    'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max',
    'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max',
    'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s',
    'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
    'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std',
    'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean',
    'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Len',
    'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Max',
    'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'Down/Up Ratio',
    'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg',
    'Subflow Fwd Pkts', 'Subflow Fwd Byts', 'Subflow Bwd Pkts',
    'Subflow Bwd Byts', 'Init Fwd Win Byts', 'Init Bwd Win Byts',
    'Fwd Act Data Pkts', 'Idle Mean', 'Idle Max', 'Idle Min'
]

# Features for StandardScaler (low skew)
STANDARD_SCALER_FEATURES = ['Fwd Seg Size Min']

# Features to leave unscaled (binary/categorical flags)
NO_SCALING_FEATURES = [
    'Protocol', 'RST Flag Cnt', 'PSH Flag Cnt',
    'ACK Flag Cnt', 'URG Flag Cnt', 'ECE Flag Cnt'
]

os.makedirs(BASE_MODEL_DIR, exist_ok=True)
print('✅ Configuration set')
print(f'   Output directory: {BASE_MODEL_DIR}')

## 3. Load Dataset

In [ ]:
CHUNK_SIZE = 100000
MAX_ROWS   = None  # None = load all data

chunks     = []
total_rows = 0

print(f'\n📂 Loading data from: {INPUT_FILE}')
print('⏳ Reading in chunks...\n')

for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):
    if MAX_ROWS and total_rows >= MAX_ROWS:
        break
    if MAX_ROWS and total_rows + len(chunk) > MAX_ROWS:
        chunk = chunk.head(MAX_ROWS - total_rows)
    chunks.append(chunk)
    total_rows += len(chunk)
    print(f'   Chunk {i+1}: {len(chunk):,} rows | Total: {total_rows:,}')

df_full = pd.concat(chunks, ignore_index=True)
del chunks

# Drop the string Label column — we only need Label_Binary
df_full = df_full.drop(columns=['Label'], errors='ignore')

print(f'\n✅ Dataset loaded')
print(f'   Rows   : {len(df_full):,}')
print(f'   Columns: {len(df_full.columns)}')
print(f'\nRaw class distribution:')
print(df_full['Label_Binary'].value_counts())

## 4. Data Cleaning
### Remove corrupt rows (impossible negative values)
CICFlowMeter has a known bug where timestamp overflows produce **negative** values
in duration and inter-arrival time features. These rows are corrupted and must be
removed before any training.

In [ ]:
print('=' * 80)
print('🧹 REMOVING CORRUPT ROWS (impossible negative values)')
print('=' * 80)

before = len(df_full)
cols_to_check = [c for c in NON_NEGATIVE_FEATURES if c in df_full.columns]

print('\nNegative value counts per feature:')
for col in cols_to_check:
    n_bad = (df_full[col] < 0).sum()
    if n_bad > 0:
        print(f'  ⚠️  {col:<30} {n_bad:>8,} corrupt values')

corrupt_mask = (df_full[cols_to_check] < 0).any(axis=1)
df_full = df_full[~corrupt_mask].copy()

removed = before - len(df_full)
print(f'\n  Before : {before:,}')
print(f'  Removed: {removed:,} corrupt rows')
print(f'  After  : {len(df_full):,}')
print(f'\nClass distribution after cleaning:')
print(df_full['Label_Binary'].value_counts())

## 5. Prepare Features

In [ ]:
print('\n' + '=' * 80)
print('🔧 PREPARING FEATURES')
print('=' * 80)

X_full = df_full.drop(['Label_Binary'], axis=1).select_dtypes(include=[np.number])
y_full = df_full['Label_Binary']

# Remove constant features (zero variance — useless for any model)
constant_cols = [c for c in X_full.columns if X_full[c].std() < 1e-10]
if constant_cols:
    print(f'\n  Dropping {len(constant_cols)} constant features (zero variance):')
    for c in constant_cols:
        print(f'    - {c}')
    X_full = X_full.drop(columns=constant_cols)

del df_full

# Store feature count for model building
NUM_FEATURES = X_full.shape[1]

print(f'\n✅ Features shape : {X_full.shape}')
print(f'✅ Labels shape   : {y_full.shape}')
print(f'   Number of features: {NUM_FEATURES}')
print(f'\nClass distribution:')
print(y_full.value_counts())

## 6. Helper Function — Prepare Data Per Model

In [ ]:
def get_model_data_for_dl(X_full, y_full, model_name, sample_size=None,
                          test_size=0.2, random_state=42):
    """
    Prepare train/test data for a DL model.

    CORRECT ORDER:
      1. Sample (optional)
      2. Split train/test FIRST
      3. Fit scalers on TRAIN only
      4. Transform both sets with fitted scalers

    This prevents any form of data leakage.
    """
    X = X_full.copy()
    y = y_full.copy()

    # --- Optional sampling (for slow models like LSTM) ---
    if sample_size is not None and sample_size < len(X):
        print(f'   📊 Sampling {sample_size:,} rows (from {len(X):,})')
        X, _, y, _ = train_test_split(
            X, y, train_size=sample_size,
            random_state=random_state, stratify=y
        )
        print(f'   Sampled class distribution: {y.value_counts().to_dict()}')
    else:
        print(f'   📊 Using full dataset: {len(X):,} rows')

    # --- STEP 1: Split FIRST — test set is isolated immediately ---
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )
    print(f'   Train: {len(X_train):,} rows | Test: {len(X_test):,} rows')

    X_train = X_train.copy()
    X_test  = X_test.copy()

    # --- STEP 2: Fit scalers on TRAIN only, transform both ---
    print('   🔧 Scaling (fit on train only — no leakage)...')
    scalers = {}

    # PowerTransformer — handles extreme skew and negative values
    power_cols = [
        c for c in POWER_TRANSFORMER_FEATURES
        if c in X_train.columns and X_train[c].std() > 1e-10
    ]
    if power_cols:
        pt = PowerTransformer(method='yeo-johnson', standardize=True)
        pt.fit(X_train[power_cols])                         # ← train only
        X_train[power_cols] = pt.transform(X_train[power_cols])
        X_test[power_cols]  = pt.transform(X_test[power_cols])
        scalers['power'] = pt
        print(f'      PowerTransformer : {len(power_cols)} features')

    # StandardScaler — for low-skew features
    std_cols = [
        c for c in STANDARD_SCALER_FEATURES
        if c in X_train.columns and X_train[c].std() > 1e-10
    ]
    if std_cols:
        ss = StandardScaler()
        ss.fit(X_train[std_cols])                           # ← train only
        X_train[std_cols] = ss.transform(X_train[std_cols])
        X_test[std_cols]  = ss.transform(X_test[std_cols])
        scalers['standard'] = ss
        print(f'      StandardScaler   : {len(std_cols)} features')

    no_scale = [c for c in NO_SCALING_FEATURES if c in X_train.columns]
    if no_scale:
        print(f'      Unscaled         : {len(no_scale)} binary/flag features')

    print('   ✅ Scaling complete')
    return X_train, X_test, y_train, y_test, scalers


print('✅ Helper function defined')

## 7. Model Builders

In [ ]:
def build_cnn_model(input_shape):
    """
    Build 1D CNN model for network traffic classification.
    """
    model = Sequential([
        tf.keras.layers.Reshape((input_shape[0], 1), input_shape=input_shape),

        # Conv Block 1
        Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        # Conv Block 2
        Conv1D(128, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        # Conv Block 3
        Conv1D(256, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.4),

        # Dense layers
        Flatten(),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dropout(0.4),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )
    return model


def build_lstm_model(input_shape):
    """
    Build LSTM model for network traffic classification.
    """
    model = Sequential([
        tf.keras.layers.Reshape((input_shape[0], 1), input_shape=input_shape),

        # LSTM layers
        LSTM(128, return_sequences=True),
        Dropout(0.3),
        BatchNormalization(),

        LSTM(64, return_sequences=False),
        Dropout(0.3),
        BatchNormalization(),

        # Dense layers
        Dense(128, activation='relu'),
        Dropout(0.4),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )
    return model


model_builders = {
    'CNN' : build_cnn_model,
    'LSTM': build_lstm_model
}

print(f'\n✅ Defined {len(model_builders)} DL models to train')

## 8. Train All Models

In [ ]:
results       = {}
histories     = {}
saved_scalers = {}

for model_name, model_builder in model_builders.items():
    print('\n' + '=' * 80)
    print(f'🚀 TRAINING: {model_name}')
    print('=' * 80)

    model_dir = os.path.join(BASE_MODEL_DIR, model_name.lower())
    os.makedirs(model_dir, exist_ok=True)

    sample_size = MODEL_SAMPLE_SIZES.get(model_name, None)

    # Prepare data — split first, scale on train only
    X_train, X_test, y_train, y_test, scalers = get_model_data_for_dl(
        X_full, y_full, model_name, sample_size, TEST_SIZE, RANDOM_STATE
    )
    saved_scalers[model_name] = scalers

    # Build model
    print(f'\n🔨 Building {model_name} model...')
    model = model_builder((NUM_FEATURES,))
    print(f'\n📊 Model Summary:')
    model.summary()

    # Callbacks
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=os.path.join(model_dir, 'best_model.h5'),
            monitor='val_loss',
            save_best_only=True,
            verbose=1
        )
    ]

    # Train
    print(f'\n⏱️  Training {model_name}...')
    start_time = time.time()

    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VALIDATION_SPLIT,
        callbacks=callbacks,
        verbose=1
    )

    train_time = time.time() - start_time
    print(f'\n✅ Training completed in {train_time/60:.2f} minutes')

    # Evaluate
    print(f'\n🔮 Evaluating on test set...')
    test_loss, test_accuracy, test_precision, test_recall = model.evaluate(
        X_test, y_test, batch_size=BATCH_SIZE, verbose=1
    )

    # Predictions for sklearn metrics
    y_pred_proba = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
    y_pred       = (y_pred_proba > 0.5).astype(int).flatten()

    # Balanced accuracy and per-class metrics
    bal_acc  = balanced_accuracy_score(y_test, y_pred)
    f1_w     = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average='macro',    zero_division=0)

    report = classification_report(
        y_test, y_pred,
        target_names=['Benign', 'Attack'],
        output_dict=True, zero_division=0
    )
    benign_prec   = report['Benign']['precision']
    benign_recall = report['Benign']['recall']
    benign_f1     = report['Benign']['f1-score']
    attack_prec   = report['Attack']['precision']
    attack_recall = report['Attack']['recall']
    attack_f1     = report['Attack']['f1-score']

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    print(f'\n📊 Results:')
    print(f'   Loss              : {test_loss:.4f}')
    print(f'   Accuracy          : {test_accuracy:.4f}  (misleading on imbalanced data)')
    print(f'   Balanced Accuracy : {bal_acc:.4f}  ← PRIMARY METRIC')
    print(f'   F1 Macro          : {f1_macro:.4f}  ← USE THIS TOO')
    print(f'\n   Per-Class Performance:')
    print(f'   {"":20} {"Precision":>10} {"Recall":>10} {"F1":>10}')
    print(f'   {"-"*52}')
    print(f'   {"Benign":20} {benign_prec:>10.4f} {benign_recall:>10.4f} {benign_f1:>10.4f}')
    print(f'   {"Attack":20} {attack_prec:>10.4f} {attack_recall:>10.4f} {attack_f1:>10.4f}')
    print(f'\n   Confusion Matrix:')
    print(f'                      Pred Benign   Pred Attack')
    print(f'   Actual Benign      {cm[0][0]:>10,}   {cm[0][1]:>10,}')
    print(f'   Actual Attack      {cm[1][0]:>10,}   {cm[1][1]:>10,}')

    results[model_name] = {
        'accuracy':          float(test_accuracy),
        'balanced_accuracy': bal_acc,
        'precision':         float(test_precision),
        'recall':            float(test_recall),
        'f1_weighted':       f1_w,
        'f1_macro':          f1_macro,
        'benign_precision':  benign_prec,
        'benign_recall':     benign_recall,
        'attack_precision':  attack_prec,
        'attack_recall':     attack_recall,
        'loss':              float(test_loss),
        'train_time':        train_time,
        'sample_size':       len(X_train)
    }
    histories[model_name] = history

    # Save
    if SAVE_MODELS:
        model_path = os.path.join(model_dir, 'final_model.h5')
        model.save(model_path)

        if scalers:
            with open(os.path.join(model_dir, 'scalers.pkl'), 'wb') as f:
                pickle.dump(scalers, f)

        metadata = {
            'model_name':    model_name,
            'num_features':  NUM_FEATURES,
            'sample_size':   len(X_train),
            'epochs_trained': len(history.history['loss']),
            'metrics':       results[model_name],
            'train_time_minutes': train_time / 60,
            'pipeline_fixes': ['corrupt_rows_removed', 'split_before_scale']
        }
        with open(os.path.join(model_dir, 'metadata.pkl'), 'wb') as f:
            pickle.dump(metadata, f)

        with open(os.path.join(model_dir, 'history.pkl'), 'wb') as f:
            pickle.dump(history.history, f)

        print(f'\n💾 Saved → {model_dir}')

## 9. Results Summary

In [ ]:
print('\n' + '=' * 80)
print('📊 FINAL RESULTS — HONEST METRICS')
print('=' * 80)

summary_df = pd.DataFrame(results).T
summary_df = summary_df.sort_values('balanced_accuracy', ascending=False)

display_cols = [
    'balanced_accuracy', 'f1_macro',
    'benign_recall', 'attack_recall',
    'benign_precision', 'attack_precision',
    'loss', 'train_time', 'sample_size'
]

print('\nRanked by Balanced Accuracy (primary metric):')
print(summary_df[display_cols].to_string())

print('\n⚠️  NOTE: Plain accuracy is misleading on imbalanced data.')
print('    Balanced Accuracy treats both classes equally.')
print('    Benign Recall  = how many legitimate flows are correctly identified')
print('    Attack Recall  = how many attacks are correctly detected')

summary_df

## 10. Visualisation — Training History

In [ ]:
print('\n📊 Creating training history plots...')

fig, axes = plt.subplots(len(model_builders), 2,
                         figsize=(14, 6 * len(model_builders)))
if len(model_builders) == 1:
    axes = axes.reshape(1, -1)

for idx, (model_name, history) in enumerate(histories.items()):
    # Loss
    axes[idx, 0].plot(history.history['loss'],     label='Train Loss')
    axes[idx, 0].plot(history.history['val_loss'], label='Val Loss')
    axes[idx, 0].set_title(f'{model_name} — Loss', fontweight='bold')
    axes[idx, 0].set_xlabel('Epoch')
    axes[idx, 0].set_ylabel('Loss')
    axes[idx, 0].legend()
    axes[idx, 0].grid(alpha=0.3)

    # Accuracy
    axes[idx, 1].plot(history.history['accuracy'],     label='Train Accuracy')
    axes[idx, 1].plot(history.history['val_accuracy'], label='Val Accuracy')
    axes[idx, 1].set_title(f'{model_name} — Accuracy', fontweight='bold')
    axes[idx, 1].set_xlabel('Epoch')
    axes[idx, 1].set_ylabel('Accuracy')
    axes[idx, 1].legend()
    axes[idx, 1].grid(alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(BASE_MODEL_DIR, 'training_history_fixed.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f'✅ Saved → {plot_path}')
plt.show()

## 11. Visualisation — Performance Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('DL Model Performance — Fixed Pipeline (CIC-IDS-2018)',
             fontsize=14, fontweight='bold')

model_names = summary_df.index.tolist()

# Balanced Accuracy
axes[0].bar(model_names, summary_df['balanced_accuracy'], color='steelblue')
axes[0].set_title('Balanced Accuracy (Primary Metric)', fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1)
axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random baseline')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Per-class Recall
x     = np.arange(len(model_names))
width = 0.35
axes[1].bar(x - width/2, summary_df['benign_recall'], width,
            label='Benign Recall', color='steelblue')
axes[1].bar(x + width/2, summary_df['attack_recall'], width,
            label='Attack Recall', color='coral')
axes[1].set_title('Per-Class Recall (Most Important)', fontweight='bold')
axes[1].set_ylabel('Recall')
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_names)
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(alpha=0.3)

# Training time
axes[2].barh(model_names, summary_df['train_time'] / 60, color='lightgreen')
axes[2].set_title('Training Time', fontweight='bold')
axes[2].set_xlabel('Minutes')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(BASE_MODEL_DIR, 'model_comparison_fixed.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f'✅ Saved → {plot_path}')
plt.show()

## 12. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, len(model_builders),
                         figsize=(6 * len(model_builders), 5))
if len(model_builders) == 1:
    axes = [axes]

for idx, (model_name) in enumerate(model_builders.keys()):
    # Re-generate predictions
    sample_size = MODEL_SAMPLE_SIZES.get(model_name, None)
    X_tr, X_te, y_tr, y_te, _ = get_model_data_for_dl(
        X_full, y_full, model_name, sample_size, TEST_SIZE, RANDOM_STATE
    )
    # Load best saved model
    best_path = os.path.join(BASE_MODEL_DIR, model_name.lower(), 'best_model.h5')
    best_model = tf.keras.models.load_model(best_path)
    y_pred_p   = best_model.predict(X_te, batch_size=BATCH_SIZE, verbose=0)
    y_pred     = (y_pred_p > 0.5).astype(int).flatten()

    cm  = confusion_matrix(y_te, y_pred)
    bal = balanced_accuracy_score(y_te, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Benign', 'Attack'],
                yticklabels=['Benign', 'Attack'])
    axes[idx].set_title(f'{model_name}\nBal.Acc={bal:.4f}',
                        fontweight='bold')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')

plt.suptitle('Confusion Matrices — DL Models', fontsize=14, fontweight='bold')
plt.tight_layout()
cm_path = os.path.join(BASE_MODEL_DIR, 'confusion_matrices_fixed.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
print(f'✅ Saved → {cm_path}')
plt.show()

## 13. Final Summary

In [ ]:
print('\n' + '=' * 80)
print('🎉 TRAINING COMPLETE — FIXED PIPELINE')
print('=' * 80)

print('\n📋 What was fixed in this notebook:')
print('   ✅ Corrupt rows removed  (impossible negative values from CICFlowMeter bug)')
print('   ✅ Scaling done correctly (split first → fit on train only)')
print('   ✅ Honest metrics used   (Balanced Accuracy + per-class Recall)')

print('\n📊 Models Ranked by Balanced Accuracy:')
for rank, (model_name, row) in enumerate(
        sorted(results.items(),
               key=lambda x: x[1]['balanced_accuracy'],
               reverse=True), 1):
    print(f'   {rank}. {model_name:<6}  '
          f'Bal.Acc={row["balanced_accuracy"]:.4f}  '
          f'Benign Recall={row["benign_recall"]:.4f}  '
          f'Attack Recall={row["attack_recall"]:.4f}  '
          f'({row["train_time"]/60:.1f} min)')

print(f'\n📁 All models saved to: {BASE_MODEL_DIR}')
print('\n💡 Key reminder:')
print('   A good IDS model needs HIGH recall on BOTH classes.')
print('   Low Benign Recall → too many false alarms.')
print('   Low Attack Recall → attacks are slipping through.')